**Extract Gravestone Text with Gemini**

This notebook transcribes every grave in the cemetery with Gemini 3.1 Pro. It reads the photos in `./data/<section>/Row N/Stone M`, for every section folder except `raw`, and sends each grave to Gemini as one request: all of the grave's photos with the extraction prompt, the answer constrained to the response JSON schema (both from Notion › Scratchpad › OCR Extraction Pipeline). The requests go through the Batch API, which costs half the interactive price and answers within 24 hours. Answers are cached per grave and prompt version, so a re-run sends only the graves still without an answer and never pays twice for the same one. The photos are read but never modified, and the notebook writes only the CSV and its working folder `./extraction`:

1. `./extracted_text.csv`: one row per person commemorated, with the grave's `section`, `row` and `stone`, the `extract_time`, and every field of the response schema
2. `./extraction/cache/<prompt version>/<section>/<row>/<stone>.json`: each grave's answer, with its photos, model, batch job and token usage
3. `./extraction/batches`: each batch job's record and raw output, kept for auditing

**1. Configuration and Helpers**

This cell defines the settings, the prompt and the functions that every later cell reuses. Before the first run, create a file `.env` next to the notebook holding the line `GEMINI_API_KEY=<your key>`: `.env` is git-ignored, and the key is read from it into `GEMINI_API_KEY`, so it never appears in the notebook. A `GEMINI_API_KEY` variable already set in the environment takes precedence. Set `STONE_LIMIT` to a small number for a cheap trial run: each run sends at most that many graves, and the next run carries on with the rest.

1. `MODEL`, `THINKING_LEVEL`, `PROMPT` and `RESPONSE_SCHEMA`: what each request asks of Gemini. `PROMPT_VERSION` fingerprints all four, so changing any of them starts a fresh answer cache
2. `CSV_COLUMNS`: the grave's location and extraction time, then every field of the response schema in schema order
3. `find_stones`: every stone folder as a `Stone`, with its photos in a format Gemini reads (JPEG, PNG, WebP, HEIC and HEIF) and any other files it skips
4. `location_fields` and `cache_path`: a grave's CSV location columns and the file that caches its answer, both derived from its key, e.g. `Women/Row 1/Stone 2`
5. `make_client`: a Gemini client that retries rate-limit and server errors

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# Each stone folder, DATA_DIR/<section>/Row N/Stone M/, holds every photo of
# one grave, so each grave is one Gemini request: the extraction prompt and
# all of the grave's photos, with the answer constrained to RESPONSE_SCHEMA.
# The requests go through the Batch API, at half the interactive price:
#   • photos are first uploaded to the Gemini File API, since ~6 GB of
#     images cannot be written inline into a batch file
#   • each answer is cached per grave and prompt version, so a re-run only
#     sends the graves still without one (new, rejected or re-prompted)
#   • a submitted job is recorded on disk, so an interrupted notebook
#     resumes waiting for it instead of paying for a second job
# ─────────────────────────────────────────────────────────────────────────────
import csv
import hashlib
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import UTC, datetime, timedelta
from pathlib import Path

import httpx
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
ENV_FILE       = Path(".env")                  # git-ignored; holds GEMINI_API_KEY=<key>
MODEL          = "gemini-3.1-pro-preview"      # Gemini 3.1 Pro
THINKING_LEVEL = "HIGH"                        # LOW, MEDIUM or HIGH (the model default)
STONE_LIMIT    = None                          # most graves sent per run; None sends all
DATA_DIR       = Path("data")                  # one folder per cemetery section
SKIPPED_DIRS   = {"raw"}                       # folders of DATA_DIR that are not sections
OUTPUT_CSV     = Path("extracted_text.csv")    # one row per person commemorated
WORK_DIR       = Path("extraction")            # uploads, batch jobs and cached answers
UPLOADS_FILE   = WORK_DIR / "uploads.json"     # each photo's upload on the File API
JOB_FILE       = WORK_DIR / "active_job.json"  # the submitted job, until collected
BATCHES_DIR    = WORK_DIR / "batches"          # each collected job's record and output
CACHE_DIR      = WORK_DIR / "cache"            # one answer per prompt version and grave
RETRY_ATTEMPTS = 5                             # tries per API call before it fails
MAX_LISTED     = 10                            # problem paths printed per report line
RULE           = "─" * 72                      # frames every printed report

# The Gemini API key is a secret, so it is never written in the notebook. It
# comes from the environment variable GEMINI_API_KEY, which load_dotenv sets
# from ENV_FILE when that file exists; a variable already set in the shell wins.
load_dotenv(ENV_FILE)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

# Image formats Gemini reads, by file extension; it reads Apple's HEIC/HEIF
# directly, so the photos are sent without conversion.
MIME_TYPES = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png",
              ".webp": "image/webp", ".heic": "image/heic", ".heif": "image/heif"}

# ═════════════════════════════════════════════════════════════════════════════
# PROMPT AND RESPONSE SCHEMA  —  from Notion › Scratchpad › OCR Extraction
# ═════════════════════════════════════════════════════════════════════════════
# A backslash at the end of a line joins it to the next one, so every
# paragraph of the prompt stays a single line while the code stays narrow.
PROMPT = """\
You will be given all the photographs taken of a single grave in a Jewish agricultural-colony \
cemetery in Entre Ríos, Argentina. The inscriptions may be in Spanish, Hebrew or both. The \
photographs show the same grave from different angles and distances. Some show the whole \
stone. Others are close-ups of the inscription, the back of the stone, a portrait or a \
separate plaque. Some may be blurred, partly hidden or show worn text. Treat every image as a \
view of the same grave and use them together to produce the most complete and accurate reading \
you can. Ignore any handwritten index card or label that is not part of the grave.

A grave may commemorate more than one person, for example a shared stone for a married couple. \
Work in two steps:

Step 1. Transcribe. Read all the text on the grave. Where the same text appears in several \
images, transcribe it once, using the clearest view to resolve it.

Step 2. Label. For each person commemorated, find the fields below in your transcription.

Return a JSON array with one object per person commemorated, in the order they appear on the \
grave. If no person can be identified, return a single object with every name, date and age \
field (including "death_date_he") set to null. Each object has exactly these fields:

"all_text": A plain-text transcription of all the text on the grave, in approximate reading \
order (top to bottom; left to right for Spanish, right to left for Hebrew). Transcribe Hebrew \
in Hebrew script. Where you can see that text is present but cannot read it with confidence, \
insert [...]. Do not add labels, headings, punctuation or anything else that is not on the \
grave. This field is identical in every object.
"given_name_es": The person's first name or names, as written in Latin script.
"given_name_he": The person's first name or names, as written in Hebrew script.
"surname_es": The person's surname, as written in Latin script.
"surname_he": The person's surname, as written in Hebrew script.
"maiden_name_es", "maiden_name_he": A married woman's maiden surname, as written in Latin and \
Hebrew script respectively (e.g. "Gold" in "Sara Gold de Kaplan", or after לבית).
"father_name_es": The father's name, as written in Latin script (e.g. after "hijo de" or "hija \
de").
"father_name_he": The father's name, as written in Hebrew script (e.g. after בן or בת).
"husband_name_es", "husband_name_he": The husband's name, as written in Latin and Hebrew script \
respectively (e.g. after "esposa de" or "viuda de", or after אשת).
"birth_day", "birth_month", "birth_year": The civil (Gregorian) date of birth, as integers.
"death_day", "death_month", "death_year": The civil (Gregorian) date of death, as integers.
"death_date_he": The Hebrew-calendar date of death, transcribed exactly as written, usually in \
Hebrew script with Hebrew-letter numerals (e.g. ט"ו אדר תשי"ב). Leave out introductory words \
such as נפטר, נפטרה or נלב"ע.
"age_of_death": The age at death in whole years, as an integer, converting Hebrew-letter \
numerals (e.g. "a los 72 años" or בת ע"ב שנה). Use 0 for an age given only in months or days.
"epitaph": Every other inscription about this person, such as dedications, eulogies, verses \
and blessings (e.g. "Q.E.P.D.", "Recuerdo de sus hijos", תנצב"ה), in the original language and \
script. Leave out the names, dates and ages already captured in other fields.
"confidence": An integer between 0 and 100 giving your best estimate of the proportion of the \
text about this person that you have transcribed correctly. 0 means no confidence and 100 \
means complete confidence. Take into account image quality, surface wear, partial occlusion \
and any text given as [...].

Follow these rules:
- Record only what is written on the grave. Never translate or transliterate between scripts. \
If a name appears only in Spanish, its Hebrew field is null, and the reverse.
- Use null for any field whose information does not appear on the grave. If the information is \
present but partly illegible, transcribe what you can and insert [...] for the unreadable \
part.
- Leave titles, honorifics and formulaic abbreviations out of the name fields (e.g. Sr., Sra., \
ר', מרת, הבחור, ז"ל, ע"ה). They stay in "all_text".
- For a married woman, record the surname exactly as written, including any "de <husband's \
surname>" form, and her maiden surname in the maiden name fields when the grave gives it. Fill \
the husband name fields only when the husband is named, not from a "de <surname>" form alone.
- Civil and Hebrew dates are recorded separately, each only if it is written on the grave. \
Never convert a date from one calendar to the other. Do not work out a birth year from an age \
at death, or an age at death from the dates.
- Civil date fields are integers: the month is a number from 1 to 12 (convert Spanish month \
names) and the year has four digits. If a civil date part cannot be read with confidence, set \
it to null instead of guessing. In "death_date_he", mark unreadable parts with [...].
- On a shared grave, give each person the epitaph text that refers to them. Text that refers \
to everyone goes in every object.

Return only valid JSON. Do not include commentary, markdown or code fences."""

# Gemini's answer must match this JSON Schema: one object per person.
RESPONSE_SCHEMA = {
    "type": "array",
    "minItems": 1,
    "items": {
        "type": "object",
        "properties": {
            "all_text":        {"type": "string"},
            "given_name_es":   {"type": ["string", "null"]},
            "given_name_he":   {"type": ["string", "null"]},
            "surname_es":      {"type": ["string", "null"]},
            "surname_he":      {"type": ["string", "null"]},
            "maiden_name_es":  {"type": ["string", "null"]},
            "maiden_name_he":  {"type": ["string", "null"]},
            "father_name_es":  {"type": ["string", "null"]},
            "father_name_he":  {"type": ["string", "null"]},
            "husband_name_es": {"type": ["string", "null"]},
            "husband_name_he": {"type": ["string", "null"]},
            "birth_day":       {"type": ["integer", "null"], "minimum": 1, "maximum": 31},
            "birth_month":     {"type": ["integer", "null"], "minimum": 1, "maximum": 12},
            "birth_year":      {"type": ["integer", "null"]},
            "death_day":       {"type": ["integer", "null"], "minimum": 1, "maximum": 31},
            "death_month":     {"type": ["integer", "null"], "minimum": 1, "maximum": 12},
            "death_year":      {"type": ["integer", "null"]},
            "death_date_he":   {"type": ["string", "null"]},
            "age_of_death":    {"type": ["integer", "null"], "minimum": 0, "maximum": 130},
            "epitaph":         {"type": ["string", "null"]},
            "confidence":      {"type": "integer", "minimum": 0, "maximum": 100},
        },
        "required": ["all_text", "given_name_es", "given_name_he", "surname_es",
                     "surname_he", "maiden_name_es", "maiden_name_he", "father_name_es",
                     "father_name_he", "husband_name_es", "husband_name_he", "birth_day",
                     "birth_month", "birth_year", "death_day", "death_month", "death_year",
                     "death_date_he", "age_of_death", "epitaph", "confidence"],
        "additionalProperties": False,
    },
}

# The CSV gives where the grave is and when it was read, then every field of
# the answer in schema order.
CSV_COLUMNS = ["section", "row", "stone", "extract_time",
               *RESPONSE_SCHEMA["items"]["properties"]]


def make_prompt_version(model: str, thinking_level: str, prompt: str,
                        schema: dict) -> str:
    """Return the first 12 hex digits of the SHA-256 hash of the four settings.

    Answers are cached under this fingerprint, so changing any setting that
    shapes Gemini's answer starts a fresh cache; older answers stay on disk.
    """
    settings = json.dumps([model, thinking_level, prompt, schema],
                          sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(settings.encode("utf-8")).hexdigest()[:12]


PROMPT_VERSION = make_prompt_version(MODEL, THINKING_LEVEL, PROMPT, RESPONSE_SCHEMA)

# ═════════════════════════════════════════════════════════════════════════════
# STONES  —  one grave per stone folder, addressed by its path in DATA_DIR
# ═════════════════════════════════════════════════════════════════════════════
@dataclass
class Stone:
    """One grave: a stone folder and the photos inside it.

    Attributes:
        section, row, name: the section, row and stone folder names, e.g.
            `Women`, `Row 2B` and `Stone 3`.
        photos: the grave's photos in a format Gemini reads, in name order.
        skipped: the folder's other files and subfolders, which are not sent.
    """

    section: str
    row: str
    name: str
    photos: list[Path] = field(default_factory=list)
    skipped: list[Path] = field(default_factory=list)

    @property
    def key(self) -> str:
        """Return the grave's path in DATA_DIR, e.g. `Women/Row 1/Stone 2`.

        The key identifies the grave in the batch file and in the cache.
        """
        return f"{self.section}/{self.row}/{self.name}"


def natural_key(name: str) -> list[int | str]:
    """Return a sort key for `name` that puts `Row 2` before `Row 10`.

    The name is split into runs of digits, compared as numbers, and runs of
    other characters, compared as lower-case text. Splitting on a captured
    pattern always alternates the two, so a number never meets text.
    """
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]


def list_entries(folder: Path) -> list[Path]:
    """Return the files and subfolders of `folder` in natural order.

    Hidden files, such as macOS's `.DS_Store`, are left out.
    """
    entries = [path for path in folder.iterdir() if not path.name.startswith(".")]
    return sorted(entries, key=lambda path: natural_key(path.name))


def read_stone(section: str, row: str, stone_dir: Path) -> Stone:
    """Return the grave in `stone_dir`, inside the given section and row.

    Every file in a format Gemini reads is a photo; anything else, including
    a subfolder, is skipped.
    """
    stone = Stone(section, row, stone_dir.name)
    for path in list_entries(stone_dir):
        if path.is_file() and path.suffix.lower() in MIME_TYPES:
            stone.photos.append(path)
        else:
            stone.skipped.append(path)
    return stone


def find_stones(data_dir: Path, skipped_dirs: set[str]) -> list[Stone]:
    """List every grave of every section.

    Args:
        data_dir: the folder holding one folder per cemetery section.
        skipped_dirs: names of folders in `data_dir` that are not sections.

    Returns:
        One `Stone` per stone folder, in section, row and stone order.
    """
    stones = []
    for section_dir in list_entries(data_dir):
        if not section_dir.is_dir() or section_dir.name in skipped_dirs:
            continue
        for row_dir in filter(Path.is_dir, list_entries(section_dir)):
            for stone_dir in filter(Path.is_dir, list_entries(row_dir)):
                stones.append(read_stone(section_dir.name, row_dir.name, stone_dir))
    return stones


def location_fields(key: str) -> dict[str, str]:
    """Split a grave's key, e.g. `Women/Row 2B/Stone 3`, into location columns.

    Returns the section in lower case, and the row and stone without their
    `Row ` and `Stone ` prefixes (`women`, `2B`, `3`). Other names stay whole.
    """
    section, row, stone = key.split("/")
    return {"section": section.lower(),
            "row": row.removeprefix("Row "),
            "stone": stone.removeprefix("Stone ")}

# ═════════════════════════════════════════════════════════════════════════════
# FILES AND CLIENT  —  JSON on disk, the answer cache and the Gemini client
# ═════════════════════════════════════════════════════════════════════════════
def read_json(path: Path, default):
    """Return the JSON value in `path`, or `default` if the file is missing."""
    if not path.is_file():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value) -> None:
    """Write `value` to `path` as indented UTF-8 JSON, creating its folders.

    The JSON goes to a temporary file that then replaces `path`, so an
    interrupted run never leaves a half-written file behind.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def cache_path(cache_dir: Path, prompt_version: str, key: str) -> Path:
    """Return `cache_dir/<prompt_version>/<section>/<row>/<stone>.json`."""
    return cache_dir / prompt_version / f"{key}.json"


def utc_text(moment: datetime) -> str:
    """Return `moment` as ISO 8601 text in UTC, e.g. `2026-10-05T14:03:12Z`."""
    return moment.astimezone(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")


def make_client(api_key: str) -> genai.Client:
    """Return a Gemini API client that uses `api_key`.

    The client retries a call that hits a rate limit or a server error up to
    RETRY_ATTEMPTS times, waiting longer before each new try.

    Raises:
        RuntimeError: if `api_key` is empty, because no key was found.
    """
    if not api_key:
        raise RuntimeError(f"No Gemini API key: add the line GEMINI_API_KEY=<your key> to "
                           f"{ENV_FILE.resolve()}, then re-run.")
    retries = types.HttpRetryOptions(attempts=RETRY_ATTEMPTS)
    return genai.Client(api_key=api_key,
                        http_options=types.HttpOptions(retry_options=retries))


print(f"✓ Configuration and helpers loaded ({MODEL}, prompt version {PROMPT_VERSION}).")

✓ Configuration and helpers loaded (gemini-3.1-pro-preview, prompt version ae2df40e60aa).


**2. Stone Inventory**

This cell takes `DATA_DIR` and, without calling Gemini or changing anything, reports for each section its rows, stones and photos, with a `[WARN]` line for every grave without photos and every file that is not sent. It then decides what this run sends:

1. `all_stones`: every grave of every section, in section, row and stone order
2. `pending_stones`: the graves with photos but no cached answer for the current `PROMPT_VERSION`
3. `batch_stones`: the pending graves this run sends, at most `STONE_LIMIT` of them, or none while a submitted job still waits to be collected

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Stone inventory  —  every grave, and which of them still need an answer
#
# Read-only pass over DATA_DIR. A grave is pending when it has photos but
# the cache holds no answer for it under the current PROMPT_VERSION. Graves
# without photos, and files that are not photos, are listed but never sent.
# STONE_LIMIT then caps the graves sent by this run, which keeps a trial run
# cheap and a large run within the Batch API's quota of queued tokens.
# ─────────────────────────────────────────────────────────────────────────────
if "find_stones" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def is_answered(stone: Stone) -> bool:
    """Return True if the cache holds `stone`'s answer for PROMPT_VERSION."""
    return cache_path(CACHE_DIR, PROMPT_VERSION, stone.key).is_file()


all_stones = find_stones(DATA_DIR, SKIPPED_DIRS)
pending_stones = [stone for stone in all_stones if stone.photos and not is_answered(stone)]

# A submitted job's graves stay pending until the 'Batch Results' cell has
# collected their answers, so nothing new is sent while JOB_FILE exists.
if JOB_FILE.exists():
    batch_stones = []
else:
    batch_stones = pending_stones[:STONE_LIMIT]  # slicing up to None keeps every grave

print(f"{RULE}\nStone inventory: {DATA_DIR}/<section>/<row>/<stone>\n{RULE}")
for _section in sorted({stone.section for stone in all_stones}, key=natural_key):
    _stones = [stone for stone in all_stones if stone.section == _section]
    print(f"  {_section:<10} {len({stone.row for stone in _stones}):>3} rows  "
          f"{len(_stones):>4} stones  {sum(len(stone.photos) for stone in _stones):>5} photos")

_warnings = [f"[WARN] {stone.key}: no photos, so it is never sent"
             for stone in all_stones if not stone.photos]
_warnings += [f"[WARN] skipped {path}: not a photo Gemini reads"
              for stone in all_stones for path in stone.skipped]
for _warning in _warnings[:MAX_LISTED]:
    print(f"  {_warning}")
if len(_warnings) > MAX_LISTED:
    print(f"  ... and {len(_warnings) - MAX_LISTED} more")

print(f"{RULE}\n  {len(all_stones)} stones with "
      f"{sum(len(stone.photos) for stone in all_stones)} photos:  "
      f"{sum(1 for stone in all_stones if is_answered(stone))} answered with prompt "
      f"version {PROMPT_VERSION}, {len(pending_stones)} pending")
if JOB_FILE.exists():
    print(f"  a submitted job waits in {JOB_FILE}: this run collects it and sends nothing new")
else:
    print(f"  this run sends {len(batch_stones)} graves"
          + (f"  (STONE_LIMIT = {STONE_LIMIT})" if STONE_LIMIT is not None else ""))

────────────────────────────────────────────────────────────────────────
Stone inventory: data/<section>/<row>/<stone>
────────────────────────────────────────────────────────────────────────
  Children    19 rows   195 stones    389 photos
  Women       43 rows   599 stones   2239 photos
────────────────────────────────────────────────────────────────────────
  794 stones with 2628 photos:  555 answered with prompt version ae2df40e60aa, 239 pending
  this run sends 239 graves


**3. Photo Upload**

This cell takes `batch_stones` and uploads their photos to the Gemini File API, since a batch request refers to its images by file URI. The File API keeps an upload for 48 hours, so a photo uploaded by an earlier run is reused while at least `MIN_HOURS_LEFT` hours of that remain, enough to outlast a batch job. `UPLOAD_WORKERS` photos upload at once, and the first full run moves about 6 GB. Interrupting it is safe, since every finished upload is recorded and reused by the next run. It produces:

1. `uploaded_files`: each photo's upload (resource name, URI, MIME type and expiration time), keyed by its path in `DATA_DIR` and saved in `UPLOADS_FILE`
2. `ready_stones`: the graves of `batch_stones` whose photos are all uploaded. A grave with a failed upload is left for the next run

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Photo upload  —  the photos of this run's graves, to the Gemini File API
#
# Each photo is uploaded once and every request refers to it by file URI.
# The File API deletes an upload after 48 hours and a job may run for 24:
#   • uploads are recorded in UPLOADS_FILE as they finish, and a photo
#     uploaded earlier is reused while MIN_HOURS_LEFT hours remain
#   • a failed upload does not stop the others; its grave waits a run
# ─────────────────────────────────────────────────────────────────────────────
if "batch_stones" not in globals():
    raise RuntimeError("Run the 'Stone Inventory' cell first.")

UPLOAD_WORKERS = 8   # photos uploaded at the same time
MIN_HOURS_LEFT = 36  # lifetime an earlier upload needs left to be reused
SAVE_EVERY     = 50  # uploads between saves of UPLOADS_FILE (and progress lines)

# The errors that mean one upload failed: an API error, a network error, an
# unreadable photo, or a photo the File API failed to process.
UPLOAD_ERRORS = (errors.APIError, httpx.HTTPError, OSError, RuntimeError)


@dataclass
class UploadReport:
    """What one round of uploads did, by photo key.

    Attributes:
        uploaded: photos uploaded by this run.
        reused: photos skipped because an earlier upload is still valid.
        failed: photos whose upload failed, each with its error message.
    """

    uploaded: list[str] = field(default_factory=list)
    reused: list[str] = field(default_factory=list)
    failed: dict[str, str] = field(default_factory=dict)


def photo_key(photo: Path) -> str:
    """Return `photo`'s path in DATA_DIR, e.g. `Women/Row 1/Stone 2/front.jpg`."""
    return photo.relative_to(DATA_DIR).as_posix()


def upload_photo(client: genai.Client, photo: Path) -> dict:
    """Upload `photo` to the File API, named by its key.

    Returns:
        The upload's record: its resource name, URI, MIME type, and the time
        the File API deletes it as ISO 8601 text.

    Raises:
        RuntimeError: if the File API failed to process the photo.
    """
    config = types.UploadFileConfig(mime_type=MIME_TYPES[photo.suffix.lower()],
                                    display_name=photo_key(photo))
    uploaded = client.files.upload(file=photo, config=config)
    if uploaded.state == types.FileState.FAILED:
        raise RuntimeError("the File API failed to process the photo")
    expires = uploaded.expiration_time or datetime.now(UTC) + timedelta(hours=48)
    return {"name": uploaded.name, "uri": uploaded.uri,
            "mime_type": uploaded.mime_type, "expiration_time": expires.isoformat()}


def upload_photos(client: genai.Client, photos: list[Path],
                  uploads: dict[str, dict]) -> UploadReport:
    """Upload every photo that has no reusable upload yet.

    Args:
        client: a connected Gemini client.
        photos: the photos the batch needs.
        uploads: the upload records of earlier runs, keyed by `photo_key`.
            New uploads are added to it in place, and it is saved to
            UPLOADS_FILE every SAVE_EVERY uploads and when the round ends.

    Returns:
        Which photos were uploaded, reused or failed.
    """
    report = UploadReport()
    reuse_until = datetime.now(UTC) + timedelta(hours=MIN_HOURS_LEFT)
    to_upload = []
    for photo in photos:
        upload = uploads.get(photo_key(photo))
        if upload and datetime.fromisoformat(upload["expiration_time"]) >= reuse_until:
            report.reused.append(photo_key(photo))
        else:
            to_upload.append(photo)

    with ThreadPoolExecutor(max_workers=UPLOAD_WORKERS) as pool:
        futures = {pool.submit(upload_photo, client, photo): photo_key(photo)
                   for photo in to_upload}
        try:
            for done, future in enumerate(as_completed(futures), start=1):
                key = futures[future]
                try:
                    uploads[key] = future.result()
                    report.uploaded.append(key)
                except UPLOAD_ERRORS as error:
                    report.failed[key] = str(error)
                if done % SAVE_EVERY == 0:
                    write_json(UPLOADS_FILE, uploads)
                    print(f"  {done} of {len(futures)} photos uploaded ...")
        finally:
            # On an interruption, cancel the uploads not yet started, and keep
            # every finished upload on disk so that the next run reuses it.
            pool.shutdown(cancel_futures=True)
            write_json(UPLOADS_FILE, uploads)
    return report


uploaded_files = read_json(UPLOADS_FILE, {})
ready_stones = []
if not batch_stones:
    print("No grave to send in this run: nothing is uploaded.")
else:
    _photos = [photo for stone in batch_stones for photo in stone.photos]
    print(f"Uploading the {len(_photos)} photos of {len(batch_stones)} graves "
          f"({UPLOAD_WORKERS} at a time) ...")
    _report = upload_photos(make_client(GEMINI_API_KEY), _photos, uploaded_files)
    ready_stones = [stone for stone in batch_stones
                    if not any(photo_key(photo) in _report.failed for photo in stone.photos)]

    print(f"\n{RULE}\nPhoto upload: {len(_photos)} photos of {len(batch_stones)} graves\n{RULE}")
    print(f"  uploaded:  {len(_report.uploaded):>5}")
    print(f"  reused:    {len(_report.reused):>5}   (uploaded by an earlier run, still valid)")
    print(f"  failed:    {len(_report.failed):>5}")
    for _key, _message in list(_report.failed.items())[:MAX_LISTED]:
        print(f"  !! {_key}\n        - {_message}")
    if len(_report.failed) > MAX_LISTED:
        print(f"  ... and {len(_report.failed) - MAX_LISTED} more")
    print(f"{RULE}\n  {len(ready_stones)} of {len(batch_stones)} graves ready to send"
          + ("" if len(ready_stones) == len(batch_stones)
             else "  (re-run the notebook to retry the others)"))

Uploading the 898 photos of 239 graves (8 at a time) ...

────────────────────────────────────────────────────────────────────────
Photo upload: 898 photos of 239 graves
────────────────────────────────────────────────────────────────────────
  uploaded:      0
  reused:      898   (uploaded by an earlier run, still valid)
  failed:        0
────────────────────────────────────────────────────────────────────────
  239 of 239 graves ready to send


**4. Batch Submission**

This cell takes `ready_stones` and `uploaded_files`, writes one request per grave to the batch file `REQUESTS_FILE`, uploads it and creates the batch job on `MODEL`. Each request holds `PROMPT` followed by the grave's photos, and a generation config that constrains the answer to `RESPONSE_SCHEMA`. The job is recorded in `JOB_FILE` as soon as it is created. While that file exists, this cell submits nothing and hands the recorded job to the next cell, so re-running the notebook never pays for the same graves twice. It produces:

1. `active_job`: the job's record (name, model, prompt version, submission time, and each grave's photos), or `None` when there is nothing to collect
2. a report of the graves, photos and estimated input tokens sent, to compare against the Batch API's quota

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Batch submission  —  one request per grave, submitted as one batch job
#
# Each line of the batch file is {"key": <grave>, "request": <request>}; the
# key comes back with the answer, whatever order answers return in. The
# request is a generateContent request in the REST API's JSON (camelCase)
# form. Temperature and media resolution keep Gemini 3's defaults: 1.0, and
# the full 1,120 tokens per image.
# ─────────────────────────────────────────────────────────────────────────────
if "ready_stones" not in globals():
    raise RuntimeError("Run the 'Photo Upload' cell first.")

REQUESTS_FILE = WORK_DIR / "requests.jsonl"  # the batch file of the latest job
IMAGE_TOKENS  = 1120                         # input tokens per photo, for the estimate


def build_request(stone: Stone, uploads: dict[str, dict]) -> dict:
    """Build the generateContent request for one grave.

    Args:
        stone: the grave, with every photo uploaded.
        uploads: the upload records, keyed by `photo_key`.

    Returns:
        The request as a JSON-ready dictionary: PROMPT, then the photos.
    """
    parts = [{"text": PROMPT}]
    for photo in stone.photos:
        upload = uploads[photo_key(photo)]
        parts.append({"fileData": {"fileUri": upload["uri"],
                                   "mimeType": upload["mime_type"]}})
    return {
        "contents": [{"role": "user", "parts": parts}],
        "generationConfig": {
            "responseMimeType": "application/json",
            "responseJsonSchema": RESPONSE_SCHEMA,
            "thinkingConfig": {"thinkingLevel": THINKING_LEVEL},
        },
    }


def submit_batch(client: genai.Client, stones: list[Stone], uploads: dict[str, dict],
                 display_name: str) -> str:
    """Write the batch file, upload it and create the batch job that runs it.

    Args:
        client: a connected Gemini client.
        stones: the graves to send, each with every photo uploaded.
        uploads: the upload records, keyed by `photo_key`.
        display_name: the job's name in the Gemini API, also given to the file.

    Returns:
        The job's resource name, e.g. `batches/123456`.
    """
    lines = [json.dumps({"key": stone.key, "request": build_request(stone, uploads)},
                        ensure_ascii=False) for stone in stones]
    REQUESTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    REQUESTS_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
    uploaded = client.files.upload(
        file=REQUESTS_FILE,
        config=types.UploadFileConfig(display_name=display_name, mime_type="jsonl"))
    job = client.batches.create(
        model=MODEL, src=uploaded.name,
        config=types.CreateBatchJobConfig(display_name=display_name))
    return job.name


active_job = read_json(JOB_FILE, None)
if active_job is not None:
    print(f"Batch job {active_job['name']} (submitted {active_job['submitted_at']}, "
          f"{len(active_job['stones'])} graves) has not been collected yet: "
          "nothing new is submitted.")
elif not ready_stones:
    print("No grave is ready to send: no batch job is submitted.")
else:
    _submitted_at = datetime.now(UTC)
    _display_name = f"gauchos-ocr-{PROMPT_VERSION}-{_submitted_at:%Y%m%d-%H%M%S}"
    _job_name = submit_batch(make_client(GEMINI_API_KEY), ready_stones, uploaded_files,
                             _display_name)
    active_job = {
        "name": _job_name,
        "display_name": _display_name,
        "model": MODEL,
        "prompt_version": PROMPT_VERSION,
        "submitted_at": utc_text(_submitted_at),
        "stones": {stone.key: [photo.name for photo in stone.photos]
                   for stone in ready_stones},
    }
    write_json(JOB_FILE, active_job)

    # The estimate counts IMAGE_TOKENS per photo and a token per four
    # characters of the prompt, to compare with the Batch API's token quota.
    _photo_count = sum(len(stone.photos) for stone in ready_stones)
    _tokens = _photo_count * IMAGE_TOKENS + len(ready_stones) * len(PROMPT) // 4
    print(f"{RULE}\nBatch submission: {_job_name}\n{RULE}")
    print(f"  model:          {MODEL}  (prompt version {PROMPT_VERSION})")
    print(f"  graves:         {len(ready_stones):>9,}   (one request each)")
    print(f"  photos:         {_photo_count:>9,}")
    print(f"  input tokens:   {_tokens:>9,}   (estimated)")
    print(f"  batch file:     {REQUESTS_FILE}  ({REQUESTS_FILE.stat().st_size / 1e6:.1f} MB)")
    print(f"{RULE}\n  job recorded in {JOB_FILE}; the next cell waits for its answers")

────────────────────────────────────────────────────────────────────────
Batch submission: batches/801oj6prqp9yvifpqzhdpbh63ezp612ze9yc
────────────────────────────────────────────────────────────────────────
  model:          gemini-3.1-pro-preview  (prompt version ae2df40e60aa)
  graves:               239   (one request each)
  photos:               898
  input tokens:   1,314,966   (estimated)
  batch file:     extraction/requests.jsonl  (1.8 MB)
────────────────────────────────────────────────────────────────────────
  job recorded in extraction/active_job.json; the next cell waits for its answers


**5. Batch Results**

This cell takes `active_job`, checks its state every `POLL_SECONDS` seconds until it finishes, then downloads its output and caches each grave's answer under the job's prompt version. Most jobs finish well within the Batch API's 24-hour target. The kernel can be interrupted while it waits: the job stays recorded in `JOB_FILE`, and re-running the notebook resumes waiting for it. An answer is accepted only if:

1. the request succeeded and the model finished normally (not cut off by its token limit or a safety filter)
2. the answer is valid JSON with the shape of `RESPONSE_SCHEMA`: a list of one object per person, each with exactly the schema's fields. The values themselves are left to the accuracy check on the hand-transcribed sample

A rejected grave stays uncached, so the next run sends it again. The job's record and raw output are kept in `BATCHES_DIR`, and `JOB_FILE` is removed so that the next run can submit a new job.

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# Batch results  —  wait for the job, then cache each grave's answer
#
# Each output line is {"key": <grave>, "response": <response>} or {"key":
# <grave>, "error": <status>}. Constrained decoding makes the answer follow
# RESPONSE_SCHEMA, so only its shape is checked here, to keep a cut-off or
# malformed answer out of the CSV. Answers are cached under the job's prompt
# version, which differs from PROMPT_VERSION if the prompt changed meanwhile.
# ─────────────────────────────────────────────────────────────────────────────
if "active_job" not in globals():
    raise RuntimeError("Run the 'Batch Submission' cell first.")

POLL_SECONDS = 60  # seconds between checks of the job's state

# The states in which a batch job has stopped for good.
FINISHED_STATES = {"JOB_STATE_SUCCEEDED", "JOB_STATE_PARTIALLY_SUCCEEDED",
                   "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}

# The fields of each person in an answer, in schema order.
ANSWER_FIELDS = list(RESPONSE_SCHEMA["items"]["properties"])


@dataclass
class CollectReport:
    """What collecting one job's output did.

    Attributes:
        cached: the graves whose answer was accepted and cached.
        persons: the number of persons in the cached answers.
        rejected: the graves without an accepted answer, each with the reason.
    """

    cached: list[str] = field(default_factory=list)
    persons: int = 0
    rejected: dict[str, str] = field(default_factory=dict)


def wait_for_job(client: genai.Client, name: str) -> types.BatchJob:
    """Check the batch job `name` every POLL_SECONDS until it finishes.

    Every change of state is printed with its time. Returns the finished job.
    """
    job = client.batches.get(name=name)
    print(f"  {utc_text(datetime.now(UTC))}  {job.state.name}")
    while job.state.name not in FINISHED_STATES:
        time.sleep(POLL_SECONDS)
        previous_state = job.state.name
        job = client.batches.get(name=name)
        if job.state.name != previous_state:
            print(f"  {utc_text(datetime.now(UTC))}  {job.state.name}")
    return job


def shape_problem(answer) -> str:
    """Return why `answer` lacks the shape of RESPONSE_SCHEMA, or "" if it has it.

    The shape is a non-empty list of one object per person, each with exactly
    the schema's fields, holding text, a number or null. The values are left
    to the accuracy check on the hand-transcribed sample.
    """
    if not isinstance(answer, list) or not answer:
        return "the answer is not a non-empty list"
    for number, person in enumerate(answer, start=1):
        if not isinstance(person, dict) or set(person) != set(ANSWER_FIELDS):
            return f"person {number} does not have exactly the schema's fields"
        if any(isinstance(value, (list, dict)) for value in person.values()):
            return f"person {number} has a field that is not text, a number or null"
    return ""


def read_answer(result: dict) -> tuple[list | None, str]:
    """Accept or reject the answer in one line of a job's output.

    Args:
        result: one parsed output line, holding a `response` or an `error`.

    Returns:
        The answer (a list of one object per person) and an empty string, or
        None and the reason the answer was rejected.
    """
    if "error" in result:
        status = result["error"]
        message = status.get("message", status) if isinstance(status, dict) else status
        return None, f"request failed: {message}"
    candidates = result.get("response", {}).get("candidates") or [{}]
    if candidates[0].get("finishReason") != "STOP":
        return None, f"no complete answer (finish reason {candidates[0].get('finishReason')})"
    parts = candidates[0].get("content", {}).get("parts", [])
    text = "".join(part.get("text", "") for part in parts if not part.get("thought"))
    try:
        answer = json.loads(text)
    except json.JSONDecodeError as error:
        return None, f"the answer is not valid JSON ({error})"
    problem = shape_problem(answer)
    return (None, problem) if problem else (answer, "")


def cache_answers(output_text: str, job_record: dict, extract_time: str) -> CollectReport:
    """Cache every accepted answer in one job's output.

    Args:
        output_text: the job's output file, one JSON object per line.
        job_record: the job's record from JOB_FILE: its name, model, prompt
            version and each grave's photos.
        extract_time: when the job finished, stored with every answer.

    Returns:
        The graves cached, their number of persons, and the graves rejected.
        A grave of the job missing from the output counts as rejected.
    """
    report = CollectReport()
    for line in filter(str.strip, output_text.splitlines()):
        result = json.loads(line)
        key = result.get("key", "")
        if key not in job_record["stones"]:
            report.rejected[key or "(no key)"] = "not a grave of this job"
            continue
        answer, problem = read_answer(result)
        if answer is None:
            report.rejected[key] = problem
            continue
        write_json(cache_path(CACHE_DIR, job_record["prompt_version"], key), {
            "key": key,
            "photos": job_record["stones"][key],
            "model": job_record["model"],
            "model_version": result["response"].get("modelVersion"),
            "prompt_version": job_record["prompt_version"],
            "batch_job": job_record["name"],
            "extract_time": extract_time,
            "usage": result["response"].get("usageMetadata"),
            "persons": answer,
        })
        report.cached.append(key)
        report.persons += len(answer)
    for key in job_record["stones"]:
        if key not in report.cached and key not in report.rejected:
            report.rejected[key] = "no result in the job's output"
    return report


collect_report = None
if active_job is None:
    print("No batch job to collect.")
else:
    _client = make_client(GEMINI_API_KEY)
    print(f"Waiting for batch job {active_job['name']}, checked every {POLL_SECONDS} s "
          "(interrupting is safe; re-run the notebook to resume) ...")
    _job = wait_for_job(_client, active_job["name"])
    _job_id = active_job["name"].rsplit("/", maxsplit=1)[-1]
    _extract_time = utc_text(_job.end_time or datetime.now(UTC))
    if _job.dest is not None and _job.dest.file_name:
        _output = _client.files.download(file=_job.dest.file_name).decode("utf-8")
        BATCHES_DIR.mkdir(parents=True, exist_ok=True)
        (BATCHES_DIR / f"{_job_id}.jsonl").write_text(_output, encoding="utf-8")
        collect_report = cache_answers(_output, active_job, _extract_time)
    else:
        collect_report = CollectReport(rejected={
            key: f"the job ended in {_job.state.name} without output"
            for key in active_job["stones"]})

    # Keep the job's record next to its output, then clear JOB_FILE so that
    # the next run is free to submit a new job.
    write_json(BATCHES_DIR / f"{_job_id}.json",
               {**active_job, "state": _job.state.name, "finished_at": _extract_time,
                "error": str(_job.error) if _job.error else None})
    JOB_FILE.unlink()

    print(f"\n{RULE}\nBatch results: {active_job['name']}  ({_job.state.name})\n{RULE}")
    if _job.error:
        print(f"  !! the job reported an error\n        - {_job.error}")
    print(f"  cached:    {len(collect_report.cached):>5} graves   "
          f"{collect_report.persons:>5} persons")
    print(f"  rejected:  {len(collect_report.rejected):>5} graves"
          + ("   (re-run the notebook to send them again)" if collect_report.rejected else ""))
    for _key, _reason in list(collect_report.rejected.items())[:MAX_LISTED]:
        print(f"  !! {_key}\n        - {_reason}")
    if len(collect_report.rejected) > MAX_LISTED:
        print(f"  ... and {len(collect_report.rejected) - MAX_LISTED} more")
    print(f"{RULE}\n  answers cached in {CACHE_DIR / active_job['prompt_version']}")
    active_job = None

Waiting for batch job batches/801oj6prqp9yvifpqzhdpbh63ezp612ze9yc, checked every 60 s (interrupting is safe; re-run the notebook to resume) ...
  2026-10-07T01:06:57Z  JOB_STATE_RUNNING
  2026-10-07T01:38:01Z  JOB_STATE_SUCCEEDED

────────────────────────────────────────────────────────────────────────
Batch results: batches/801oj6prqp9yvifpqzhdpbh63ezp612ze9yc  (JOB_STATE_SUCCEEDED)
────────────────────────────────────────────────────────────────────────
  cached:      239 graves     244 persons
  rejected:      0 graves
────────────────────────────────────────────────────────────────────────
  answers cached in extraction/cache/ae2df40e60aa


**6. CSV Output**

This cell takes `all_stones` and the answer cache for the current `PROMPT_VERSION`, and writes `OUTPUT_CSV` from scratch with one row per person commemorated. The CSV is rebuilt from the cache on every run, so it holds every answer of the current prompt version, whichever job produced it. Graves are written in section, row and stone order, and the persons of a shared grave in the order the answer lists them. It reports, for each section:

1. the graves with an answer and the persons written
2. the graves still without an answer, which the next run sends

The columns are `section` (`women` or `children`), `row` and `stone` (the folder names without their `Row ` and `Stone ` prefixes), `extract_time` (when Gemini finished the batch job, in UTC), then every field of the response schema. An empty cell is a null: the field is not written on the grave.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CSV output  —  one row per person, rebuilt from the answer cache
#
# Every grave with photos is looked up in the cache under the current
# PROMPT_VERSION, so answers from several runs (a trial run, then the rest,
# then the retried graves) come together in one file. The CSV is UTF-8, and
# a multi-line inscription stays in one quoted cell.
# ─────────────────────────────────────────────────────────────────────────────
if "all_stones" not in globals():
    raise RuntimeError("Run the 'Stone Inventory' cell first.")


def person_rows(record: dict) -> list[dict]:
    """Return the CSV rows of one cached answer, one per person.

    Each row holds the grave's location columns, the answer's
    `extract_time` and the person's fields.
    """
    location = location_fields(record["key"])
    return [{**location, "extract_time": record["extract_time"], **person}
            for person in record["persons"]]


def gather_rows(stones: list[Stone]) -> tuple[list[dict], list[str]]:
    """Collect the CSV rows of every grave in `stones`, in order.

    Returns:
        The rows of every grave answered under PROMPT_VERSION, and the keys
        of the graves without an answer.
    """
    rows, unanswered = [], []
    for stone in stones:
        path = cache_path(CACHE_DIR, PROMPT_VERSION, stone.key)
        if path.is_file():
            rows += person_rows(read_json(path, None))
        else:
            unanswered.append(stone.key)
    return rows, unanswered


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` to `path` as a UTF-8 CSV with the columns CSV_COLUMNS.

    A None value is written as an empty cell.
    """
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


csv_rows, unanswered_keys = gather_rows([stone for stone in all_stones if stone.photos])
write_csv(OUTPUT_CSV, csv_rows)

print(f"{RULE}\nCSV output: {OUTPUT_CSV}  (prompt version {PROMPT_VERSION})\n{RULE}")
for _section in sorted({stone.section for stone in all_stones}, key=natural_key):
    _keys = [stone.key for stone in all_stones if stone.section == _section and stone.photos]
    _missing = [key for key in _keys if key in unanswered_keys]
    _persons = sum(1 for row in csv_rows if row["section"] == _section.lower())
    print(f"  {'!! ' if _missing else 'OK '}{_section:<10} "
          f"{len(_keys) - len(_missing):>4} of {len(_keys):>4} graves  {_persons:>5} persons"
          + (f"  | {len(_missing)} without an answer" if _missing else ""))
    for _key in _missing[:MAX_LISTED]:
        print(f"        - {_key}")
    if len(_missing) > MAX_LISTED:
        print(f"        ... and {len(_missing) - MAX_LISTED} more")
print(f"{RULE}\n  {len(csv_rows)} rows written for the graves answered so far; "
      f"{len(unanswered_keys)} graves still without an answer")

────────────────────────────────────────────────────────────────────────
CSV output: extracted_text.csv  (prompt version ae2df40e60aa)
────────────────────────────────────────────────────────────────────────
  OK Children    195 of  195 graves    198 persons
  OK Women       599 of  599 graves    627 persons
────────────────────────────────────────────────────────────────────────
  825 rows written for the graves answered so far; 0 graves still without an answer


**7. Consistency Check**

This cell rebuilds the expected CSV rows straight from the cache files of the current `PROMPT_VERSION`, independently of the previous cell, and compares them with `OUTPUT_CSV` read back from disk. It checks that:

1. the CSV's header is `CSV_COLUMNS`
2. every cached answer whose stone folder still exists is in the CSV, with one row per person, and the CSV holds no grave without a cached answer
3. every field of every row matches the cache, comparing as text with null as an empty cell, so that any value altered by the CSV round trip is caught

It reports the graves and rows of each section, and ends with the graves in the CSV against the graves with photos. Graves still without an answer leave the CSV incomplete, not inconsistent, and the next run sends them.

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the CSV against the answer cache
#
# The expected rows are rebuilt from the cache files on disk rather than
# from the previous cell's rows: every answer cached under PROMPT_VERSION
# whose stone folder still exists. Graves are identified by their location
# columns, `section/row/stone`, and compared field by field as text, with
# null as an empty cell and numbers in decimal.
# ─────────────────────────────────────────────────────────────────────────────
if "csv_rows" not in globals():
    raise RuntimeError("Run the 'CSV Output' cell first.")


def expected_graves() -> dict[str, list[dict[str, str]]]:
    """Rebuild each grave's CSV rows, as text, from the cache files on disk.

    Returns:
        Each grave, as `section/row/stone`, mapped to its rows. An answer
        whose stone folder no longer exists in DATA_DIR is left out.
    """
    expected = {}
    for path in sorted((CACHE_DIR / PROMPT_VERSION).glob("*/*/*.json")):
        record = json.loads(path.read_text(encoding="utf-8"))
        if not (DATA_DIR / record["key"]).is_dir():
            continue
        location = location_fields(record["key"])
        rows = [{**location, "extract_time": record["extract_time"], **person}
                for person in record["persons"]]
        expected["/".join(location.values())] = [
            {column: "" if row[column] is None else str(row[column]) for column in CSV_COLUMNS}
            for row in rows]
    return expected


def found_graves(csv_path: Path) -> tuple[list[str], dict[str, list[dict[str, str]]]]:
    """Read the CSV at `csv_path` back, grouping its rows by grave.

    Returns:
        The CSV's header, and each grave, as `section/row/stone`, mapped to
        its rows in file order.
    """
    found: dict[str, list[dict[str, str]]] = {}
    with csv_path.open(encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        for row in reader:
            found.setdefault(f"{row.get('section')}/{row.get('row')}/{row.get('stone')}",
                             []).append(row)
        header = list(reader.fieldnames or [])
    return header, found


def grave_problems(grave: str, expected_rows: list[dict] | None,
                   found_rows: list[dict] | None) -> list[str]:
    """Compare one grave's rows in the CSV with its cached answer.

    Args:
        grave: the grave, as `section/row/stone`.
        expected_rows: its rows rebuilt from the cache, or None if uncached.
        found_rows: its rows in the CSV, or None if absent.

    Returns:
        One message per difference, or an empty list if the rows match.
    """
    if found_rows is None:
        return [f"{grave}: cached but missing from the CSV"]
    if expected_rows is None:
        return [f"{grave}: in the CSV without a cached answer"]
    if len(found_rows) != len(expected_rows):
        return [f"{grave}: {len(found_rows)} rows in the CSV, "
                f"{len(expected_rows)} persons in the cache"]
    problems = []
    for number, (found, expected) in enumerate(zip(found_rows, expected_rows), start=1):
        for column, value in expected.items():
            if found.get(column) != value:
                problems.append(f"{grave} person {number}: {column} is "
                                f"{found.get(column)!r} in the CSV, {value!r} in the cache")
    return problems


def check_csv(csv_path: Path, graves_with_photos: int) -> bool:
    """Print a per-section consistency report of the CSV against the cache.

    Args:
        csv_path: the CSV file.
        graves_with_photos: the number of graves that can have an answer.

    Returns:
        True when the CSV holds exactly the cached answers.
    """
    header, found = found_graves(csv_path)
    expected = expected_graves()
    print(f"{RULE}\nConsistency: {csv_path} vs {CACHE_DIR / PROMPT_VERSION}\n{RULE}")
    failed = 0
    if header != CSV_COLUMNS:
        failed += 1
        print(f"  !! header      {len(header)} columns that are not CSV_COLUMNS")
    graves = sorted(set(expected) | set(found), key=natural_key)
    for section in sorted({grave.split("/")[0] for grave in graves}):
        in_section = [grave for grave in graves if grave.split("/")[0] == section]
        problems = [message for grave in in_section
                    for message in grave_problems(grave, expected.get(grave), found.get(grave))]
        failed += 1 if problems else 0
        print(f"  {'!! ' if problems else 'OK '}{section:<10} {len(in_section):>4} graves"
              f"  {sum(len(found.get(grave, [])) for grave in in_section):>5} rows"
              + (f"  | {len(problems)} problem(s)" if problems else ""))
        for message in problems[:MAX_LISTED]:
            print(f"        - {message}")
    waiting = graves_with_photos - len(found)
    print(f"{RULE}\n  {len(found)} of {graves_with_photos} graves in the CSV  ("
          + ("consistent" if failed == 0 else f"{failed} check(s) need attention")
          + (f"; {waiting} still without an answer)" if waiting else ")"))
    return failed == 0


csv_consistent = check_csv(OUTPUT_CSV, sum(1 for stone in all_stones if stone.photos))

print("\n✓ Extraction complete (one Gemini request per grave, answers cached per "
      "prompt version).")
print(f"  csv consistent: {csv_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: extracted_text.csv vs extraction/cache/ae2df40e60aa
────────────────────────────────────────────────────────────────────────
  OK children    195 graves    198 rows
  OK women       599 graves    627 rows
────────────────────────────────────────────────────────────────────────
  794 of 794 graves in the CSV  (consistent)

✓ Extraction complete (one Gemini request per grave, answers cached per prompt version).
  csv consistent: True
